In [1]:
# %% [markdown]
# CDPKit Conformer Generation & Rosetta Ranking Pipeline
# 
# This notebook demonstrates:
# 1. Pulling .params files from the Enamine REAL library
# 2. Generating 150 conformers per ligand using CDPKit
# 3. Ranking each conformer with Rosetta energy scores

# %%
import sys, os

# Point to the discovery module
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'func', 'discovery'))
from cdpkit_conformer_pipeline import (
    extract_params_from_enamine,
    generate_cdpkit_conformers,
    convert_conformer_to_params,
    score_single_conformer_with_rosetta,
)
# ## Configuration
# Adjust these paths for your setup:

# ── Input ──────────────────────────────────────────────────────────
INPUT_LIST    = "conformer_input.txt"          # shapedb results list
TARGET_PDB    = "../input_pdb/processed/orexin_lemborexant_receptor_R.pdb"  # change to your target
ANCHOR_RESIDUES = "138"                           # anchor residue(s)
MOTIFS_FILE   = "../motifs/FINAL_motifs_list_filtered_2_3_2023.motifs"

# ── Paths ──────────────────────────────────────────────────────────
REALM_LOCATION = "/pi/summer.thyme-umw/Ji_rosetta_discovery"
ENAMINE_PATH   = "/pi/summer.thyme-umw/enamine-REAL-2.6billion"
OUTPUT_DIR     = "./output/cdpkit_test"

# ── Parameters ─────────────────────────────────────────────────────
NUM_CONFORMERS = 150
TOP_N          = 20
ATR, REP, DDG  = -2.0, 150.0, -9.0

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

Output directory: ./output/cdpkit_test


In [2]:
## Step 1: Read input list
entries = []
with open(INPUT_LIST, 'r') as fh:
    for line in fh:
        print(line.rstrip())
        line = line.strip()
        if not line:
            continue
        fields = [f.strip() for f in line.split(',')]
        if len(fields) < 4:
            continue
        score, ligand_conf, chunk, subchunk = fields[:4]
        ligand_name = "_".join(ligand_conf.split('_')[:-1])
        try:
            conf_num = int(ligand_conf.split('_')[-1])
            entries.append((float(score), ligand_name, conf_num, chunk, subchunk))
        except ValueError:
            continue

for s, name, cn, ch, sc in entries:
    print(f"  {name}  conf={cn}  chunk={ch}  subchunk={sc}  score={s:.4f}")


-0.433132231236,PV-006439055682_1,33229,0
  PV-006439055682  conf=1  chunk=33229  subchunk=0  score=-0.4331


In [3]:

# %% [markdown]
# ## Step 2: Extract params from Enamine library

# %%
params_dir = os.path.join(OUTPUT_DIR, 'extracted_params')
os.makedirs(params_dir, exist_ok=True)

results = []
for score, ligand_name, conf_num, chunk, subchunk in entries:
    print(f"\n--- {ligand_name} (conf {conf_num}) ---")
    pf = extract_params_from_enamine(
        ligand_name, conf_num, chunk, subchunk,
        ENAMINE_PATH, params_dir
    )
    if pf:
        results.append((score, ligand_name, conf_num, chunk, subchunk, pf))
        print(f"  ✓ Extracted: {os.path.basename(pf)}")
    else:
        print(f"  ✗ Failed to extract params")

print(f"\nSuccessfully extracted {len(results)}/{len(entries)} params files")




--- PV-006439055682 (conf 1) ---
  ✓ Extracted: PV-006439055682_1.params

Successfully extracted 1/1 params files


In [4]:
## Step 3: Extract SMILES from params

from rdkit import Chem

# %%
from cdpkit_conformer_pipeline  import (
    extract_smiles_from_params
)

smiles_map = {}
for score, ligand_name, conf_num, chunk, subchunk, pf in results:
    with open(pf, 'r') as fh:
        params_text = fh.read()
    
    smiles = extract_smiles_from_params(params_text)
    if smiles:
        smiles_map[ligand_name] = smiles
        print(f"  {ligand_name}: {smiles}")
    else:
        print(f"  {ligand_name}: ✗ Could not extract SMILES")


    Parsed 47 atoms, 50 bonds from params
    Generated SMILES via RDKit: [H]OC([H])([H])C([H])(C([H])([H])c1c([H])nc([H])c([H])c1[H])C([H])([H])N([H])C(=O)c1c([H])c([H])c([H])c2c1oc1c([H])c([H])c([H])c([H])c12
  PV-006439055682: [H]OC([H])([H])C([H])(C([H])([H])c1c([H])nc([H])c([H])c1[H])C([H])([H])N([H])C(=O)c1c([H])c([H])c([H])c2c1oc1c([H])c([H])c([H])c([H])c12


In [6]:

# %% [markdown]
# ## Step 4: Generate conformers with CDPKit

# %%
all_conformers = {}
for ligand_name, smiles in smiles_map.items():
    print(f"\nGenerating {NUM_CONFORMERS} conformers for {ligand_name}...")
    conformers = generate_cdpkit_conformers(smiles, NUM_CONFORMERS)
    if conformers:
        all_conformers[ligand_name] = conformers
        print(f"  ✓ Generated {len(conformers)} conformers")
    else:
        print(f"  ✗ Failed to generate conformers")

print(f"\nGenerated conformers for {len(all_conformers)}/{len(smiles_map)} ligands")



Generating 150 conformers for PV-006439055682...
ERROR: CDPKit conformer generation failed: module 'CDPL.Chem' has no attribute 'addHydrogens'
  ✗ Failed to generate conformers

Generated conformers for 0/1 ligands


In [ ]:

# %% [markdown]
# ## Step 5: Convert conformers to Rosetta .params

# %%
conf_params_dir = os.path.join(OUTPUT_DIR, 'conformer_params')
os.makedirs(conf_params_dir, exist_ok=True)

all_params = {}  # ligand_name -> [(conf_idx, params_path), ...]
for ligand_name, conformers in all_conformers.items():
    all_params[ligand_name] = []
    for i, conf_mol in enumerate(conformers):
        from cdpkit_conformer_pipeline import write_cdpkit_conformer_to_sdf
        sdf_path = os.path.join(conf_params_dir, f"{ligand_name}_{i+1}.sdf")
        write_cdpkit_conformer_to_sdf(conf_mol, sdf_path)
        
        pf = convert_conformer_to_params(
            sdf_path, ligand_name, i+1, conf_params_dir, REALM_LOCATION
        )
        if pf:
            all_params[ligand_name].append((i+1, pf))
    print(f"  {ligand_name}: {len(all_params[ligand_name])} params files")



In [ ]:
# %% [markdown]
# ## Step 6: Rank conformers with Rosetta

# %%
import heapq

global_ranked = []
score_dir = os.path.join(OUTPUT_DIR, 'rosetta_scores')
os.makedirs(score_dir, exist_ok=True)

residue = ANCHOR_RESIDUES.split(',')[0].strip()

for ligand_name, conf_list in all_params.items():
    print(f"\nScoring {len(conf_list)} conformers for {ligand_name}...")
    for conf_idx, pf in conf_list:
        conf_name = f"{ligand_name}_{conf_idx}"
        work_subdir = os.path.join(score_dir, conf_name)
        os.makedirs(work_subdir, exist_ok=True)
        
        conf_scores = score_single_conformer_with_rosetta(
            pf, TARGET_PDB, residue, MOTIFS_FILE,
            REALM_LOCATION, ATR, REP, DDG, work_subdir
        )
        
        if conf_scores:
            best_conf_score = max(conf_scores.values())
            entry = (-best_conf_score, ligand_name, conf_idx, best_conf_score)
            if len(global_ranked) < TOP_N:
                heapq.heappush(global_ranked, entry)
            elif best_conf_score > -global_ranked[0][0]:
                heapq.heapreplace(global_ranked, entry)
            print(f"    {conf_name}: score={best_conf_score:.4f}")
        else:
            print(f"    {conf_name}: no scores")



In [ ]:
# %% [markdown]
# ## Results: Top Ranked Conformers

# %%
sorted_results = sorted(global_ranked, reverse=True)
max_score = max(s[3] for s in sorted_results) if sorted_results else 1.0

print(f"{'Rank':<6} {'Ligand':<30} {'Conf#':<6} {'Score':<12} {'Normalized'}")
print("-" * 70)
for rank, (neg_score, ligand_name, conf_idx, conf_score) in enumerate(sorted_results, 1):
    norm = conf_score / max_score if max_score != 0 else 0
    print(f"{rank:<6} {ligand_name:<30} {conf_idx:<6} {conf_score:<12.4f} {norm:<10.4f}")

# Save to CSV
import csv
output_csv = os.path.join(OUTPUT_DIR, 'ranked_conformers.csv')
with open(output_csv, 'w', newline='') as fh:
    writer = csv.writer(fh)
    writer.writerow(['rank', 'ligand', 'conformer', 'rosetta_score', 'normalized_score'])
    for rank, (neg_score, ligand_name, conf_idx, conf_score) in enumerate(sorted_results, 1):
        norm = conf_score / max_score if max_score != 0 else 0
        writer.writerow([rank, ligand_name, conf_idx, f"{conf_score:.4f}", f"{norm:.4f}"])

print(f"\n✅ Results saved to: {output_csv}")